# Обучение и оценка YOLOv8 для детектирования трещин

Этот ноутбук демонстрирует:
- Загрузку предобученной модели YOLOv8n
- Inference на тестовых изображениях
- Визуализацию результатов
- Метрики качества: mAP, Precision, Recall

## Требования
- ultralytics >= 8.2.0
- torch >= 2.0.0
- CUDA (опционально, для ускорения inference)

## 1. Импорт библиотек и настройка

In [ ]:
import os
import json
from pathlib import Path
from typing import List, Dict, Any

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image

# Настройка matplotlib для отображения изображений
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 12

# Проверка доступности GPU
print(f"PyTorch версия: {torch.__version__}")
print(f"CUDA доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Версия CUDA: {torch.version.cuda}")

## 2. Определение путей к данным

In [ ]:
# Получение корневой директории проекта
PROJECT_ROOT = Path.cwd().parent

# Пути к данным и моделям
DATASETS_DIR = PROJECT_ROOT / 'datasets' / 'defects'
MODELS_DIR = PROJECT_ROOT / 'models' / 'yolo'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

# Датасет SDNET2018
SDNET_DIR = DATASETS_DIR / 'sdnet2018'
SDNET_YOLO_DIR = SDNET_DIR / 'yolo'

# Проверка существования директорий
print(f"📁 Корень проекта: {PROJECT_ROOT}")
print(f"📁 Датасеты: {DATASETS_DIR}")
print(f"📁 Модели: {MODELS_DIR}")

# Проверка наличия датасета
if not DATASETS_DIR.exists():
    print("⚠️  Директория датасетов не найдена!")
    print("Запустите: python scripts/download_datasets.py")
else:
    print("✓ Директория датасетов найдена")

## 3. Загрузка предобученной модели YOLOv8n

In [ ]:
# Загрузка предобученной YOLOv8n (nano) модели
print("Загрузка предобученной модели YOLOv8n...")
model = YOLO('yolov8n.pt')

# Информация о модели
print(f"✓ Модель загружена")
print(f"  Классы модели: {model.names}")
print(f"  Количество классов: {len(model.names)}")

# Настройка устройства
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nУстройство для inference: {device}")

## 4. Загрузка тестовых изображений

In [ ]:
def load_test_images(images_dir: Path, num_images: int = 5) -> List[Path]:
    """
    Загрузить пути к тестовым изображениям.
    
    Args:
        images_dir: Директория с изображениями.
        num_images: Количество изображений для загрузки.
    
    Returns:
        Список путей к изображениям.
    """
    if not images_dir.exists():
        print(f"⚠️  Директория не найдена: {images_dir}")
        return []
    
    # Поиск изображений
    image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
    
    if not image_files:
        print(f"⚠️  Изображения не найдены в {images_dir}")
        return []
    
    # Перемешивание и выборка
    np.random.seed(42)
    np.random.shuffle(image_files)
    
    return image_files[:num_images]


# Загрузка тестовых изображений
test_images_dir = SDNET_YOLO_DIR / 'images' / 'test'
test_images = load_test_images(test_images_dir, num_images=5)

print(f"Найдено тестовых изображений: {len(test_images)}")
for i, img_path in enumerate(test_images, 1):
    print(f"  {i}. {img_path.name}")

## 5. Inference на тестовых изображениях

In [ ]:
def run_inference(model: YOLO, image_paths: List[Path], conf_threshold: float = 0.25) -> List:
    """
    Запустить inference на списке изображений.
    
    Args:
        model: Модель YOLO.
        image_paths: Пути к изображениям.
        conf_threshold: Порог уверенности детекции.
    
    Returns:
        Список результатов inference.
    """
    results = []
    
    for img_path in image_paths:
        print(f"Обработка: {img_path.name}...")
        
        result = model.predict(
            source=str(img_path),
            conf=conf_threshold,
            iou=0.45,
            device=device,
            verbose=False
        )
        
        results.append(result[0])
    
    return results


# Запуск inference
print("Запуск inference на тестовых изображениях...\n")
inference_results = run_inference(model, test_images, conf_threshold=0.25)

print(f"\n✓ Inference завершен")

## 6. Визуализация результатов детекции

In [ ]:
def plot_detections(images: List[Path], results: List, class_names: Dict[int, str]) -> None:
    """
    Визуализировать результаты детекции.
    
    Args:
        images: Пути к изображениям.
        results: Результаты inference.
        class_names: Словарь имен классов.
    """
    fig, axes = plt.subplots(len(images), 1, figsize=(15, 8 * len(images)))
    
    if len(images) == 1:
        axes = [axes]
    
    for idx, (ax, img_path, result) in enumerate(zip(axes, images, results), 1):
        # Загрузка изображения
        img = Image.open(img_path)
        ax.imshow(img)
        
        # Получение bounding boxes
        boxes = result.boxes
        
        if boxes is not None and len(boxes) > 0:
            # Визуализация детекций
            result_img = result.plot()
            ax.imshow(result_img)
            
            # Информация о детекциях
            det_info = f"Найдено объектов: {len(boxes)}\n"
            for box in boxes:
                cls = int(box.cls[0])
                conf = float(box.conf[0])
                class_name = class_names.get(cls, f"class_{cls}")
                det_info += f"  • {class_name}: {conf:.2%}\n"
            
            ax.text(
                0.02, 0.98, det_info,
                transform=ax.transAxes,
                fontsize=10,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8)
            )
        else:
            ax.text(
                0.5, 0.5, 'Объекты не обнаружены',
                transform=ax.transAxes,
                fontsize=14,
                ha='center',
                va='center',
                bbox=dict(boxstyle='round', facecolor='red', alpha=0.5)
            )
        
        ax.set_title(f'Изображение {idx}: {img_path.name}', fontsize=14, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()


# Визуализация результатов
# Примечание: YOLOv8n предобучена на COCO (80 классов)
# Для детектирования трещин потребуется fine-tuning
print("Визуализация результатов детекции...\n")
plot_detections(test_images, inference_results, model.names)

## 7. Метрики модели (после fine-tuning)

In [ ]:
def load_metrics(metrics_path: Path) -> Dict[str, Any]:
    """
    Загрузить метрики из JSON файла.
    
    Args:
        metrics_path: Путь к файлу с метриками.
    
    Returns:
        Словарь с метриками.
    """
    if not metrics_path.exists():
        print(f"⚠️  Файл с метриками не найден: {metrics_path}")
        return {}
    
    with open(metrics_path, 'r', encoding='utf-8') as f:
        return json.load(f)


def plot_metrics(train_metrics: Dict, eval_metrics: Dict) -> None:
    """
    Визуализировать метрики обучения.
    
    Args:
        train_metrics: Метрики обучения.
        eval_metrics: Метрики оценки.
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Основные метрики
    metrics_to_plot = {
        'mAP@50': eval_metrics.get('mAP50', 0),
        'mAP@50-95': eval_metrics.get('mAP50-95', 0),
        'Precision': eval_metrics.get('precision', 0),
        'Recall': eval_metrics.get('recall', 0)
    }
    
    # График 1: Основные метрики
    ax1 = axes[0, 0]
    names = list(metrics_to_plot.keys())
    values = list(metrics_to_plot.values())
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
    
    bars = ax1.bar(names, values, color=colors, alpha=0.8)
    ax1.set_ylabel('Значение', fontsize=12)
    ax1.set_title('Метрики качества модели', fontsize=14, fontweight='bold')
    ax1.set_ylim(0, 1)
    ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
    
    # Подписи значений
    for bar, value in zip(bars, values):
        ax1.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{value:.3f}',
            ha='center',
            va='bottom',
            fontsize=11
        )
    
    # График 2: Параметры обучения
    ax2 = axes[0, 1]
    params = {
        'Эпохи': train_metrics.get('epochs', 0),
        'Batch Size': train_metrics.get('batch_size', 0),
        'Image Size': train_metrics.get('image_size', 0)
    }
    
    param_names = list(params.keys())
    param_values = list(params.values())
    
    ax2.bar(param_names, param_values, color=['#9b59b6', '#1abc9c', '#34495e'])
    ax2.set_ylabel('Значение', fontsize=12)
    ax2.set_title('Параметры обучения', fontsize=14, fontweight='bold')
    
    # Подписи значений
    for bar, value in zip(ax2.patches, param_values):
        ax2.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + max(param_values)*0.01,
            str(value),
            ha='center',
            va='bottom',
            fontsize=11
        )
    
    # График 3: Функция потерь
    ax3 = axes[1, 0]
    losses = {
        'Box Loss': train_metrics.get('box_loss', 0),
        'Class Loss': train_metrics.get('cls_loss', 0),
        'DFL Loss': train_metrics.get('dfl_loss', 0)
    }
    
    loss_names = list(losses.keys())
    loss_values = list(losses.values())
    
    ax3.bar(loss_names, loss_values, color=['#e74c3c', '#f39c12', '#16a085'])
    ax3.set_ylabel('Loss', fontsize=12)
    ax3.set_title('Функции потерь (финальные)', fontsize=14, fontweight='bold')
    
    # Подписи значений
    for bar, value in zip(ax3.patches, loss_values):
        ax3.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{value:.4f}',
            ha='center',
            va='bottom',
            fontsize=10
        )
    
    # График 4: Fitness
    ax4 = axes[1, 1]
    fitness = train_metrics.get('final_fitness', 0)
    
    colors_fitness = ['#e74c3c' if fitness < 0.3 else '#f39c12' if fitness < 0.6 else '#2ecc71']
    ax4.bar(['Fitness'], [fitness], color=colors_fitness)
    ax4.set_ylabel('Значение', fontsize=12)
    ax4.set_title(f'Fitness модели: {fitness:.4f}', fontsize=14, fontweight='bold')
    ax4.set_ylim(0, 1)
    
    # Подпись значения
    ax4.text(0, fitness + 0.05, f'{fitness:.4f}', ha='center', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()


# Загрузка метрик
model_name = 'yolov8n'
train_metrics_path = MODELS_DIR / model_name / 'train_metrics.json'
eval_metrics_path = MODELS_DIR / model_name / 'eval_metrics.json'

train_metrics = load_metrics(train_metrics_path)
eval_metrics = load_metrics(eval_metrics_path)

if train_metrics and eval_metrics:
    print("✓ Метрики загружены")
    print(f"\n📊 Основные метрики:")
    print(f"  mAP@50:    {eval_metrics.get('mAP50', 0):.4f}")
    print(f"  mAP@50-95: {eval_metrics.get('mAP50-95', 0):.4f}")
    print(f"  Precision: {eval_metrics.get('precision', 0):.4f}")
    print(f"  Recall:    {eval_metrics.get('recall', 0):.4f}")
    
    # Визуализация
    plot_metrics(train_metrics, eval_metrics)
else:
    print("⚠️  Метрики не найдены. Сначала запустите обучение:")
    print("   python scripts/train_yolo.py --dataset sdnet2018 --epochs 100")

## 8. Запуск fine-tuning (опционально)

In [ ]:
from IPython.display import display, Markdown

display(Markdown("""
### Для запуска fine-tuning YOLOv8n на датасете трещин:

**Вариант 1: Через скрипт**
```bash
python scripts/train_yolo.py --dataset sdnet2018 --epochs 100 --batch 16
```

**Вариант 2: Прямо в ноутбуке**
```python
from ultralytics import YOLO

# Загрузка модели
model = YOLO('yolov8n.pt')

# Обучение
results = model.train(
    data='datasets/defects/sdnet2018/dataset.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    project='models/yolo',
    name='yolov8n_crack'
)
```

**Параметры для экспериментов:**
- `--model yolov8s.pt` - большая модель (small)
- `--epochs 200` - больше эпох
- `--batch 32` - больший batch (если есть память GPU)
- `--img-size 512` - меньший размер для скорости
"""))

## 9. Сохранение обученной модели

In [ ]:
def save_model_for_inference(model: YOLO, save_path: Path) -> None:
    """
    Сохранить модель для последующего использования.
    
    Args:
        model: Обученная модель.
        save_path: Путь для сохранения.
    """
    save_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Сохранение в формате PyTorch
    model.export(format='torchscript')
    
    # Копирование best.pt
    best_model = MODELS_DIR / model_name / 'weights' / 'best.pt'
    if best_model.exists():
        import shutil
        shutil.copy2(best_model, save_path)
        print(f"✓ Модель сохранена: {save_path}")
    else:
        print("⚠️  Обученная модель не найдена. Запустите обучение.")


# Сохранение модели
final_model_path = MODELS_DIR / 'crack_detection_best.pt'
save_model_for_inference(model, final_model_path)

## 10. Сводка результатов

## 📋 Сводка

### Выполненные действия:
1. ✅ Загружена предобученная модель YOLOv8n
2. ✅ Загружены тестовые изображения из SDNET2018
3. ✅ Выполнен inference на 5 изображениях
4. ✅ Визуализированы результаты детекции
5. ✅ Показаны метрики качества (mAP, Precision, Recall)

### Следующие шаги:
- Запустить fine-tuning на полном датасете
- Оптимизировать гиперпараметры
- Протестировать на реальных изображениях

### Информация о запуске:

In [ ]:
from datetime import datetime

print(f"📅 Дата выполнения: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"💻 Устройство: {device}")
print(f"📁 Модель сохранена: {final_model_path}")